In [102]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [103]:
words = open("files/names.txt").read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [104]:
len(words)

32033

In [105]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0 
itos = {i:s for s,i in stoi.items()}

In [106]:
block_size = 3 # context lenght: how many characters do we take to predict the next one     
X,Y = [], []
for w in words:

    # print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print(context)
        # print(''.join(itos[i] for i in context), "--->",itos[ix])
        context = context[1:] + [ix] # crop and update
        
X = torch.tensor(X)
Y = torch.tensor(Y)


In [107]:
X.shape,X.dtype,Y.shape,Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [108]:
C = torch.randn((27,2))

In [109]:
C[5]

tensor([-0.4977, -1.9262])

In [110]:
F.one_hot(torch.tensor(5),27).float() @ C

tensor([-0.4977, -1.9262])

In [111]:
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [112]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [114]:
emb * W1 + b1

RuntimeError: The size of tensor a (2) must match the size of tensor b (100) at non-singleton dimension 2

In [115]:
torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1).shape

torch.Size([228146, 6])

In [116]:
len(torch.unbind(emb,1))

3

In [117]:
torch.cat(torch.unbind(emb,1),1).shape

torch.Size([228146, 6])

In [118]:
emb.view(32,6)[0] == torch.cat(torch.unbind(emb,1),1)[0]

RuntimeError: shape '[32, 6]' is invalid for input of size 1368876

In [119]:
h = torch.tanh(emb.view(32,6) @ W1 + b1)
h,h.shape

RuntimeError: shape '[32, 6]' is invalid for input of size 1368876

In [120]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)
logits = h @ W2 + b2
logits.shape

torch.Size([32, 27])

In [121]:
counts = logits.exp()

In [122]:
probs = counts / counts.sum(1,True)
probs.shape

torch.Size([32, 27])

In [123]:
probs[torch.arange(32),Y]

IndexError: shape mismatch: indexing tensors could not be broadcast together with shapes [32], [228146]

In [124]:
loss = -probs[torch.arange(32),Y].log().mean()
loss

IndexError: shape mismatch: indexing tensors could not be broadcast together with shapes [32], [228146]

In [125]:
# ---------- More Respected ----------------

In [126]:
X.shape,Y.shape # dataset

(torch.Size([228146, 3]), torch.Size([228146]))

In [138]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2),generator=g)
W1 = torch.randn((6,100),generator=g)
b1 = torch.randn(100,generator=g)
W2 = torch.randn((100,27),generator=g)
b2 = torch.randn(27,generator=g)
parameters = [C,W1,b1,W2,b2]

In [139]:
sum(p.nelement() for p in parameters)

3481

In [140]:
# Forward Pass
emb = C[X] # 32,3,2
h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32,100)
logits = h @ W2 + b2

# counts = logits.exp()
# probs = counts/counts.sum(1,True)
# loss = -probs[torch.arange(32),Y].log().mean()

loss = F.cross_entropy(logits,Y)
loss

tensor(19.5052)

In [141]:
for p in parameters:
    p.requires_grad = True

In [142]:
lre = torch.linspace(-3,1,1000)
lrs = 10**lre


In [ ]:
lri = []
lossi = []
for i in range(100):

    # minibatch construct
    ix = torch.randint(0,X.shape[0],(32,))

    # Forward Pass
    emb = C[X[ix]] # 32,3,2
    h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32,100)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits,Y[ix])
    

    # Backward Pass

    for p in parameters:
        p.grad = None

    loss.backward()

    lr = lrs[i]
    for p in parameters:
        p.data += -lr * p.grad

    lri.append(lr)
    lossi.append(loss.item())

    # print(loss.item())


In [147]:
loss

tensor(16.4127, grad_fn=<NllLossBackward0>)

In [ ]:
torch.randint(0,X.shape[0],(32,))

tensor([194957,  73255, 195733,  36876,  35493,  95288, 125281,  75164,  83296,
        140384, 219922,  27410,  58657,  55730, 188185, 187584, 165659, 128304,
        132027, 132688,  48920, 132798, 177252, 103164, 160281,    294, 152927,
        223712,  56802,  20670,   2291, 162895])

In [ ]:
emb.shape

torch.Size([228146, 3, 2])